# 02 — Carga da Camada Silver

Este notebook lê a Origins (Bronze) e produz a Silver com limpeza, padronização e integração.

**Transformações aplicadas**:
- Limpeza e padronização de nomes e tipos
- Tratamento de valores ausentes e registros inválidos
- Deduplicação por chaves de negócio
- Validação de consistência e chaves de relacionamento
- **Integração das bases**: join Indicador + UF + Município + Metas

**Regras de qualidade** embutidas com relatório ao final.

**Próximo passo**: executar `03_carga_camada_gold.py`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

tabelas_bronze = [
    "origens.tc02_uf",
    "origens.tc02_meta_brasil",
    "origens.tc02_meta_uf",
    "origens.tc02_meta_mun",
    "origens.tc02_alunos",
    "origens.tc02_municipio",
    "origens.tc02_dicionario",
]

for tabela in tabelas_bronze:
    try:
        spark.read.table(tabela).limit(1).count()
    except Exception as err:
        raise ValueError(
            f"Tabela Bronze '{tabela}' não encontrada. Execute 02_carga_camada_bronze.py."
        ) from err

print("Pré-requisitos verificados. Iniciando carga Silver...")

## Leitura das tabelas Bronze

In [0]:
def retornaDataFrame(nomeTabela, dropMeta=True, apenasUltimaParticao=True):
    _drop_meta = ["_data_ingestao_bronze", "_fonte", "_sistema_origem", "_data_criacao_origem"]

    df = spark.read.table(nomeTabela)

    if apenasUltimaParticao:
        # Pega o par (ano, mes) mais recente com base na própria coluna de partição
        ultima = (
            df.select("_ano_ingestao", "_mes_ingestao")
              .distinct()
              .orderBy(F.col("_ano_ingestao").desc(), F.col("_mes_ingestao").desc())
              .first()
        )
        if ultima is not None:
            df = df.filter(
                (F.col("_ano_ingestao") == ultima["_ano_ingestao"]) &
                (F.col("_mes_ingestao") == ultima["_mes_ingestao"])
            )

    if dropMeta:
        df = df.drop(*_drop_meta)

    return df


df_bz_uf          = retornaDataFrame("origens.tc02_uf")
df_bz_meta_brasil = retornaDataFrame("origens.tc02_meta_brasil")
df_bz_meta_uf     = retornaDataFrame("origens.tc02_meta_uf")
df_bz_meta_mun    = retornaDataFrame("origens.tc02_meta_mun")
df_bz_alunos      = retornaDataFrame("origens.tc02_alunos")
df_bz_dicionario  = retornaDataFrame("origens.tc02_dicionario")
df_bz_municipio   = retornaDataFrame("origens.tc02_municipio")

In [0]:
# Input padrão de normalização de valores nulos para as colunas de percentuais alunos por nível
input_percentual_zero_proporcao_alunos_nivel = {
    "proporcao_aluno_nivel_0": 0.0,
    "proporcao_aluno_nivel_1": 0.0,
    "proporcao_aluno_nivel_2": 0.0,
    "proporcao_aluno_nivel_3": 0.0,
    "proporcao_aluno_nivel_4": 0.0,
    "proporcao_aluno_nivel_5": 0.0,
    "proporcao_aluno_nivel_6": 0.0,
    "proporcao_aluno_nivel_7": 0.0,
    "proporcao_aluno_nivel_8": 0.0
}

## Silver 1: Dimensão UF

In [0]:
df_silver_uf = (
    df_bz_uf
    .withColumn("sigla_uf", F.trim(F.upper(F.col("sigla_uf"))))
    .withColumn("serie", F.trim(F.col("serie")))
    .withColumn("rede", F.trim(F.col("rede")))
    .fillna(value=input_percentual_zero_proporcao_alunos_nivel)
    .dropna(subset=["ano", "sigla_uf", "serie", "rede"])
    .dropDuplicates(["ano", "sigla_uf", "serie", "rede"])
    .withColumn("_data_processamento", F.current_timestamp())
    .withColumn("media_portugues", F.round(F.col("media_portugues"), 2))
)

print(f"Silver UF: {df_silver_uf.count()} registros")
display(df_silver_uf)

## Silver 2: Meta Brasil

In [0]:
df_silver_meta_brasil = (
    df_bz_meta_brasil
    .withColumn("rede", F.trim(F.upper(F.col("rede"))))
    .withColumn("ano", F.col("ano").cast("int"))
    .dropna(subset=["ano", "rede"])
    .dropDuplicates(["ano", "rede"])
    .filter(
        (F.col("taxa_alfabetizacao").between(0, 100)) &
        (F.col("percentual_participacao").between(0, 100))
    )
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Meta Brasil: {df_silver_meta_brasil.count()} registros")
display(df_silver_meta_brasil)


## Silver 3: Meta UF

In [0]:
df_silver_meta_uf = (
    df_bz_meta_uf
    .withColumn("sigla_uf", F.trim(F.upper(F.col("sigla_uf"))))
    .withColumn("rede", F.trim(F.upper(F.col("rede"))))
    .withColumn("ano", F.col("ano").cast("int"))
    .dropna(subset=["ano", "sigla_uf", "rede"])
    .dropDuplicates(["ano", "sigla_uf", "rede"])
    .filter(
        (F.col("taxa_alfabetizacao").between(0, 100)) &
        (F.col("percentual_participacao").between(0, 100))
    )
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Meta UF: {df_silver_meta_uf.count()} registros")
display(df_silver_meta_uf)
    

## Silver 4: Meta Município

In [0]:
df_silver_meta_mun = (
    df_bz_meta_mun
    .withColumn("id_municipio", F.trim(F.col("id_municipio")))
    .withColumn("rede", F.trim(F.upper(F.col("rede"))))
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("nivel_alfabetizacao", F.col("nivel_alfabetizacao").cast("int"))
    .dropna(subset=["ano", "id_municipio", "rede"])
    .dropDuplicates(["ano", "id_municipio", "rede"])
    .filter(
        (F.col("taxa_alfabetizacao").between(0, 100)) &
        (F.col("percentual_participacao").between(0, 100))
    )
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Meta Município: {df_silver_meta_mun.count()} registros")
display(df_silver_meta_mun)

## Silver 5: Dimensão Alunos

In [0]:
df_silver_alunos = (
    df_bz_alunos
    .withColumnRenamed("peso_aluno", "peso_prova_portugues")
    .withColumn("id_municipio", F.trim(F.col("id_municipio")))
    .withColumn("id_escola", F.trim(F.col("id_escola")))
    .withColumn("id_aluno", F.trim(F.col("id_aluno")))
    .withColumn("caderno", F.trim(F.upper(F.col("caderno"))))
    .withColumn("serie", F.trim(F.upper(F.col("serie"))))
    .withColumn("rede", F.trim(F.upper(F.col("rede"))))
    .withColumn("presenca", F.trim(F.upper(F.col("presenca"))))
    .withColumn("preenchimento_caderno", F.trim(F.upper(F.col("preenchimento_caderno"))))
    .withColumn("alfabetizado", F.trim(F.upper(F.col("alfabetizado"))))
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("proficiencia", F.round(F.col("proficiencia"), 2))
    .dropna(subset=["ano", "id_aluno"])
    .dropDuplicates(["ano", "id_aluno"])
    .filter(
        (F.col("peso_prova_portugues") > 0) &
        (F.col("proficiencia") >= 0)
    )
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Alunos: {df_silver_alunos.count()} registros")
display(df_silver_alunos)

## Silver 6: Dimensão Município

In [0]:
df_silver_municipio = (
    df_bz_municipio
    .withColumn("id_municipio", F.trim(F.col("id_municipio")))
    .withColumn("serie", F.trim(F.upper(F.col("serie"))))
    .withColumn("rede", F.trim(F.upper(F.col("rede"))))
    .withColumn("media_portugues", F.round(F.col("media_portugues"), 2))
    .withColumn("ano", F.col("ano").cast("int"))
    .dropna(subset=["ano", "id_municipio", "serie", "rede"])
    . fillna(value=input_percentual_zero_proporcao_alunos_nivel)
    .dropDuplicates(["ano", "id_municipio", "serie", "rede"])
    .filter(
        (F.col("taxa_alfabetizacao").between(0, 100))
    )
    .withColumn("_data_processamento", F.current_timestamp())
)

print(f"Silver Município: {df_silver_municipio.count()} registros")
display(df_silver_municipio)

In [0]:
df_silver_municipio.select("id_municipio").distinct().count()

In [0]:
df_silver_municipio.select("id_municipio").distinct().count()

## Relatório de Qualidade de Dados

Verificações aplicadas após a Silver:
- Duplicidade
- Valores ausentes em campos críticos
- Indicador fora do range esperado
- Municípios sem correspondência na dimensão
- Registros sem meta UF cadastrada

In [0]:
config_qualidade = {
    "Alunos": {
        "bronze": spark.read.table("origens.tc02_alunos"),
        "silver": df_silver_alunos,
        "pks": ["ano", "id_municipio", "id_escola", "id_aluno"],
        "cols_not_null": ["alfabetizado", "proficiencia"],
        "ranges": {}
    },
    "Meta Brasil": {
        "bronze": spark.read.table("origens.tc02_meta_brasil"),
        "silver": df_silver_meta_brasil,
        "pks": ["ano", "rede"],
        "cols_not_null": ["taxa_alfabetizacao", "meta_alfabetizacao_2024"],
        "ranges": {"taxa_alfabetizacao": (0, 100)}
    },
    "Meta Municipios": {
        "bronze": spark.read.table("origens.tc02_meta_mun"),
        "silver": df_silver_meta_mun,
        "pks": ["ano", "id_municipio", "rede"],
        "cols_not_null": ["taxa_alfabetizacao", "meta_alfabetizacao_2024"],
        "ranges": {"taxa_alfabetizacao": (0, 100)}
    },
    "Meta UF": {
        "bronze": spark.read.table("origens.tc02_meta_uf"),
        "silver": df_silver_meta_uf,
        "pks": ["ano", "sigla_uf", "rede"],
        "cols_not_null": ["taxa_alfabetizacao", "meta_alfabetizacao_2024"],
        "ranges": {"taxa_alfabetizacao": (0, 100)}
    },
    "Municipio": {
        "bronze": spark.read.table("origens.tc02_municipio"),
        "silver": df_silver_municipio,
        "pks": ["ano", "id_municipio", "serie", "rede"],
        "cols_not_null": ["taxa_alfabetizacao", "media_portugues"],
        "ranges": {"taxa_alfabetizacao": (0, 100)}
    },
    "UF": {
        "bronze": spark.read.table("origens.tc02_uf"),
        "silver": df_silver_uf,
        "pks": ["ano", "sigla_uf", "serie", "rede"],
        "cols_not_null": ["taxa_alfabetizacao", "media_portugues"],
        "ranges": {"taxa_alfabetizacao": (0, 100)}
    }
}

In [0]:
def gerar_relatorio_qualidade_silver(config_dict):
    for nome_dataset, params in config_dict.items():
        df_brz = params["bronze"]
        df_slv = params["silver"]
        pks = params.get("pks", [])
        cols_not_null = params.get("cols_not_null", [])
        ranges = params.get("ranges", {})
        total_bronze = df_brz.count()
        total_silver = df_slv.count()
        descartados = total_bronze - total_silver
        pct_descartados = round((descartados / total_bronze * 100), 2) if total_bronze > 0 else 0
        
        print("=" * 60)
        print(f"RELATÓRIO DE QUALIDADE — CAMADA SILVER: {nome_dataset.upper()}")
        print("=" * 60)
        print(f"Registros Bronze (bruto):   {total_bronze}")
        print(f"Registros Silver (limpo):   {total_silver}")
        print(f"Descartados na limpeza:     {descartados} ({pct_descartados}%)")
        
        # Análise de Duplicatas na origem 
        if pks and total_bronze > 0:
            unicos_bronze = df_brz.dropDuplicates(pks).count()
            duplicados = total_bronze - unicos_bronze
            print(f"Duplicatas removidas (chaves: {', '.join(pks)}): {duplicados}")
            
        # Análise de Nulos (Camada Silver)
        if cols_not_null and total_silver > 0:
            print(f"\n--- Validação de Completude (Silver) ---")
            for col in cols_not_null:
                nulos = df_slv.filter(F.col(col).isNull()).count()
                pct_nulos = round((nulos / total_silver * 100), 2)
                print(f"Nulos em '{col}': {nulos} ({pct_nulos}%)")
                
        # Análise de Regras de Negócio (Limites de Valores)
        if ranges and total_silver > 0:
            print(f"\n--- Validação de Limites / Range (Silver) ---")
            for col, (min_val, max_val) in ranges.items():
                fora_range = df_slv.filter((F.col(col) < min_val) | (F.col(col) > max_val)).count()
                pct_fora = round((fora_range / total_silver * 100), 2)
                print(f"Valores fora do range [{min_val}, {max_val}] em '{col}': {fora_range} ({pct_fora}%)")
                
        print("\n")

gerar_relatorio_qualidade_silver(config_qualidade)
print("=" * 60)
print("QUALIDADE VERIFICADA — Lote pronto para Camada Gold")
print("=" * 60)

## Escrita na Camada Silver

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

tabelas_silver = {
    "silver.tc02_dim_uf":                df_silver_uf,
    "silver.tc02_dim_municipio":         df_silver_municipio,
    "silver.tc02_meta_brasil":           df_silver_meta_brasil,
    "silver.tc02_meta_uf":               df_silver_meta_uf,
    "silver.tc02_meta_municipio":        df_silver_meta_mun,
    "silver.tc02_alunos":                df_silver_alunos,
}

for nome, df in tabelas_silver.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("_ano_ingestao", "_mes_ingestao")
        .saveAsTable(nome)
    )

print("Tabelas Silver criadas com sucesso:")
for nome in tabelas_silver:
    print(f"  - {nome} => {spark.read.table(nome).count()} linhas")

print("\nPróximo passo: executar 04_carga_camada_gold.py")

In [0]:
spark.sql("DESCRIBE DETAIL silver.tc02_dim_uf").select("partitionColumns").show(truncate=False)